# Seed v0: brains that grow, learn and evolve on their own

**What this is.** A first experiment in an AI architecture that is never trained with labels. There are three loops:

1. **Evolution (generations).** A population of tiny genomes (155 numbers each) improves over generations.
2. **Development (birth).** Each genome *grows* a brain from wiring rules. It holds rules, not weights, and siblings come out slightly different.
3. **Learning (lifetime).** There's no teacher and no backprop. Synapses change by a local Hebbian rule, gated by a "that mattered" neuromodulator that reacts to felt energy change and to surprise. A local prediction loop learns to guess the next sensation.

**The world.** A small 2D world with two kinds of blobs, A and B. In every new world, a hidden rule decides which is food and which is poison. No inborn instinct can beat chance, so **the only way to win is to learn during life.** Agents live in families of four relatives, see each other, and can emit a signal the others can sense. Energy runs out and agents die.

**How to run it.** Go to *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. The full run (two experiments) takes roughly 20–40 minutes on a free T4; the first log line shows the speed. Set `QUICK = True` below for a 2-minute smoke test.

**What we're testing.** Does evolution discover brains that *learn*? We check this two ways:
- **Accuracy rises within a life.** If food-choice accuracy in the second half of life is higher than in the first, the agents learned *this* world's rule.
- **The lesion test.** Freeze all synapses in an evolved brain. If performance collapses, plasticity was doing real work.

In [ ]:
import jax
print("JAX", jax.__version__, "| devices:", jax.devices())
if jax.devices()[0].platform == "cpu":
    print("No GPU found. Runtime -> Change runtime type -> T4 GPU (it still runs on CPU, just slower).")

## 1. The Seed engine
The whole architecture is in this one cell: world, development, brain, lifetime and evolution. Read the comments, then change things.

In [ ]:
"""Seed v0: evolving brains that learn on their own.

Three nested loops:
  evolution   (generations)  - a population of tiny genomes improves by evolution strategies
  development (birth)        - each genome GROWS a brain from wiring rules (twins differ)
  learning    (lifetime)     - no labels, no backprop: local Hebbian plasticity gated by a
                               neuromodulator, plus a local prediction loop (surprise signal)

The world: a small torus grid with two kinds of blobs (A and B). In every new world a hidden
rule decides which one is food and which is poison, so no fixed instinct can win -- agents
must LEARN within their lifetime. Agents live in families (same genome), see each other,
and can emit a signal that others can sense.
"""
from functools import partial
from typing import NamedTuple

import numpy as np
import jax
import jax.numpy as jnp
from jax.flatten_util import ravel_pytree

"""Bounded activity-driven growth/pruning among existing neurons (default off).

Matrices use (post, pre). Birth caps never change within a lifetime. This is a
fixed engineering rule, not an evolved rule or a biologically calibrated model.
"""
import jax.numpy as jnp
import numpy as np

MODE = 'activity-grow-prune-v1'
INTERVAL = 10
PRUNE_FRACTION = 0.01
STREAM_TAG = 107


class StructuralValidationError(ValueError):
    pass


def birth_caps(mask, legal):
    return jnp.minimum(jnp.sum(mask > 0, axis=-1) + 1, legal.sum(-1))


def rewire(W, mask, activity, legal, caps, scale, draws, *, wmax,
           alive=True, plastic=True, enabled=True, event=True):
    """One pure event; draws supplies one signed normal draw per target row.

    Validation errors are explicit outputs so jit/vmap can retain them. Public
    evaluation/training boundaries must call validate_structure before use.
    """
    present = mask > 0
    active = jnp.asarray(alive) & plastic & enabled & event
    score = jnp.abs(activity[:, None] * activity[None, :])
    weak = legal & present & (score == 0) & (jnp.abs(W) <= PRUNE_FRACTION * wmax)
    prune_index = jnp.argmin(jnp.where(weak, jnp.abs(W), jnp.inf), axis=1)
    rows = jnp.arange(W.shape[0])
    prune = active & jnp.any(weak, axis=1)
    removed = jnp.zeros_like(present).at[rows, prune_index].set(prune)
    after_prune = present & ~removed
    degree = after_prune.sum(1)
    # An isolated target needs a presynaptic bootstrap; coactivity alone is zero.
    score = jnp.where((degree == 0)[:, None], jnp.abs(activity)[None, :], score)
    candidates = legal & ~present & (score > 0)
    grow_index = jnp.argmax(jnp.where(candidates, score, -jnp.inf), axis=1)
    grow = active & (degree < caps) & jnp.any(candidates, axis=1)
    proposal = draws * scale[rows, grow_index]
    invalid = grow & ~jnp.isfinite(proposal)
    proposal = jnp.clip(proposal, -wmax, wmax)
    grow = grow & ~invalid & (proposal != 0)
    added = jnp.zeros_like(present).at[rows, grow_index].set(grow)
    new_mask = (after_prune | added).astype(mask.dtype)
    new_W = jnp.where(removed, 0., W)
    new_W = new_W.at[rows, grow_index].set(jnp.where(grow, proposal, new_W[rows, grow_index]))
    return new_W, new_mask, dict(
        added=jnp.sum((new_mask > 0) & ~present),
        removed=jnp.sum(present & (new_mask == 0)), invalid_proposals=invalid.sum())


def validate_structure(diagnostics):
    """Reject failed structural validation before accepting fitness or ES updates."""
    for key in ('invalid_proposals', 'cap_violations', 'illegal_edges', 'invalid_weights'):
        if key in diagnostics and np.any(np.asarray(diagnostics[key]) != 0):
            raise StructuralValidationError(f'structural validation failed: {key}={np.asarray(diagnostics[key]).sum()}')



class Cfg(NamedTuple):
    G: int = 16          # world is G x G (wraps around)
    M: int = 4           # agents per world (one family)
    T: int = 300         # lifetime in steps
    R: int = 2           # view radius -> (2R+1)^2 patch
    n_items: int = 24    # blobs of each kind
    H: int = 48          # hidden neurons
    n_htypes: int = 3    # hidden neuron types (genome sets rules per type pair)
    e0: float = 1.0      # starting energy
    decay: float = 0.003 # energy cost per step of being alive
    gain: float = 0.25   # energy from food (lost from poison)
    e_max: float = 2.0
    micro: int = 2       # neural updates per world step
    wmax: float = 3.0    # synapse weight bound


def dims(c):
    V = 2 * c.R + 1
    D = 4 * V * V + 3          # 4 visual channels + energy + energy change + bias
    O = 6                      # 5 movement logits + 1 signal
    n = D + c.H + O
    K = 2 + c.n_htypes         # sensor, hidden types..., motor
    return V, D, O, n, K


def neuron_types(c):
    V, D, O, n, K = dims(c)
    assert c.H % c.n_htypes == 0
    ht = np.repeat(np.arange(1, 1 + c.n_htypes), c.H // c.n_htypes)
    return jnp.array(np.concatenate([np.zeros(D, int), ht, np.full(O, K - 1)]))


# ----------------------------------------------------------------------------- genome
def init_genome(c, key):
    """The genome holds RULES, not weights: ~155 numbers in total."""
    K = dims(c)[4]
    return dict(
        conn=jnp.full((K, K), -1.0),                        # wiring prob. logit (post type, pre type)
        wscale=jnp.zeros((K, K)),                           # initial synapse strength (log)
        hebb=0.1 * jax.random.normal(key, (K, K, 4)),       # plasticity rule A,B,C,D per type pair
        log_eta=jnp.array(-1.5),                            # plasticity rate (log)
        mod=jnp.array([1.0, 0.0, 0.0]),                     # neuromodulator: energy change, surprise, bias
        log_plr=jnp.array(-3.0),                            # prediction-loop learning rate
    )


def biology_prior(c, key=None):
    """A genome seeded with a rule nature already found: dopamine-gated Hebbian learning.
    Synapses from senses to movement strengthen when a recent action raised energy
    ("that was good") and weaken when it lowered it. Everything else starts neutral,
    and evolution is free to change all of it."""
    K = dims(c)[4]
    g = init_genome(c, jax.random.PRNGKey(0) if key is None else key)
    hebb = jnp.zeros((K, K, 4)).at[K - 1, 0, 0].set(1.0)   # motor <- sensor: modulator x pre x post
    return dict(g, hebb=hebb, log_eta=jnp.array(-1.0), mod=jnp.array([3.0, 0.0, 0.0]),
                conn=g["conn"].at[K - 1, 0].set(0.0))


def genome_size(c):
    return ravel_pytree(init_genome(c, jax.random.PRNGKey(0)))[0].size


# ----------------------------------------------------------------------------- development
def develop(g, key, c):
    """Grow one brain from the genome's wiring rules. Different keys -> different twins."""
    V, D, O, n, K = dims(c)
    types = neuron_types(c)
    k1, k2 = jax.random.split(key)
    tp, tq = types[:, None], types[None, :]
    P = jax.nn.sigmoid(g["conn"])[tp, tq]
    P = P * (tp != 0) * (1.0 - jnp.eye(n))                  # sensors are clamped to the world
    mask = jax.random.bernoulli(k1, P).astype(jnp.float32)
    fanin = jnp.maximum(P.sum(1, keepdims=True), 1.0)
    W = mask * jax.random.normal(k2, (n, n)) * jnp.exp(g["wscale"])[tp, tq] / jnp.sqrt(fanin)
    return W, mask


# ----------------------------------------------------------------------------- the world
def observe(grid, pos, alive, sig, energy, denergy, c):
    G, R, M = c.G, c.R, c.M
    offs = jnp.arange(-R, R + 1)
    rows = (pos[:, 0, None] + offs[None, :]) % G
    cols = (pos[:, 1, None] + offs[None, :]) % G
    amap = (grid == 1).astype(jnp.float32)
    bmap = (grid == 2).astype(jnp.float32)
    occ = jnp.zeros((G, G)).at[pos[:, 0], pos[:, 1]].add(alive)
    smap = jnp.zeros((G, G)).at[pos[:, 0], pos[:, 1]].add(alive * sig)
    patch = lambda mp: mp[rows[:, :, None], cols[:, None, :]]
    pa, pb, po, ps = patch(amap), patch(bmap), patch(occ), patch(smap)
    po = po.at[:, R, R].add(-alive)                          # don't see yourself
    ps = ps.at[:, R, R].add(-alive * sig)
    flat = lambda a: a.reshape(M, -1)
    return jnp.concatenate(
        [flat(pa), flat(pb), flat(po), flat(ps),
         energy[:, None] - 1.0, 4.0 * denergy[:, None], jnp.ones((M, 1))], axis=1)


MOVES = jnp.array([[0, 0], [-1, 0], [1, 0], [0, -1], [0, 1]])


def init_world(key, c):
    k1, k2 = jax.random.split(key)
    cells = jax.random.choice(k1, c.G * c.G, (2 * c.n_items,), replace=False)
    vals = jnp.concatenate([jnp.ones(c.n_items, jnp.int32), 2 * jnp.ones(c.n_items, jnp.int32)])
    grid = jnp.zeros(c.G * c.G, jnp.int32).at[cells].set(vals).reshape(c.G, c.G)
    pos = jax.random.randint(k2, (c.M, 2), 0, c.G)
    return grid, pos


# ----------------------------------------------------------------------------- one brain tick
def brain_step(W, x, Wp, obs, denergy, mask, hebb, eta, plr, modw, plastic, c, *, predictor_plastic=None):
    V, D, O, n, K = dims(c)
    h = x[D:D + c.H]
    # prediction loop: the brain predicted this observation last step; learn from the miss
    err = obs - Wp @ h
    surprise = jnp.mean(err ** 2)
    pp = plastic if predictor_plastic is None else predictor_plastic
    Wp = jnp.clip(Wp + pp * plr * jnp.outer(err, h), -c.wmax, c.wmax)
    # neuromodulator ("that mattered"): driven by felt energy change and surprise
    m = jnp.tanh(modw[0] * 10.0 * denergy + modw[1] * 10.0 * surprise + modw[2])
    # three-factor Hebbian update on the activity that produced the last action
    post, pre = x[:, None], x[None, :]
    dW = hebb[..., 0] * post * pre + hebb[..., 1] * pre + hebb[..., 2] * post + hebb[..., 3]
    W = jnp.clip(W + plastic * eta * m * dW * mask, -c.wmax, c.wmax)
    # think
    x = x.at[:D].set(obs)
    for _ in range(c.micro):
        x = jnp.tanh(W @ x).at[:D].set(obs)
    return W, x, Wp, surprise


# ----------------------------------------------------------------------------- one lifetime
@partial(jax.jit, static_argnames=("c", "plastic", "record", "predictor_plastic", "structural"))
def rollout(g, wkey, dkey, rule, c, plastic=True, record=False, *, predictor_plastic=None, structural=False):
    """One family lives one life in one world. rule=0: A is food; rule=1: B is food."""
    V, D, O, n, K = dims(c)
    types = neuron_types(c)
    hebb = g["hebb"][types[:, None], types[None, :]]
    eta, plr = jnp.exp(g["log_eta"]), jnp.exp(g["log_plr"])
    pl = 1.0 if plastic else 0.0
    pp = pl if predictor_plastic is None else float(predictor_plastic)
    W, mask = jax.vmap(lambda k: develop(g, k, c))(jax.random.split(dkey, c.M))
    grid, pos = init_world(wkey, c)
    food_t, poison_t = 1 + rule, 2 - rule
    M = c.M
    carry = dict(
        grid=grid, pos=pos, W=W, x=jnp.zeros((M, n)), Wp=jnp.zeros((M, D, c.H)),
        energy=jnp.full((M,), c.e0), alive=jnp.ones(M), sig=jnp.zeros(M),
        denergy=jnp.zeros(M), counts=jnp.zeros((M, 2, 2)), key=wkey, t=0)
    if structural:
        legal = (types[:, None] != 0) & ~jnp.eye(n, dtype=bool)
        caps = birth_caps(mask, legal)
        P = jax.nn.sigmoid(g['conn'])[types[:, None], types[None, :]] * legal
        scale = jnp.exp(g['wscale'])[types[:, None], types[None, :]] / jnp.sqrt(jnp.maximum(P.sum(1, keepdims=True), 1.))
        structural_keys = jax.vmap(lambda k: jax.random.fold_in(k, STREAM_TAG))(jax.random.split(dkey, M))
        edges = (mask > 0).sum((1, 2))
        zeros = jnp.zeros(M, dtype=jnp.int32)
        carry.update(mask=mask, structure=dict(
            added=zeros, removed=zeros, initial_edges=edges, final_edges=edges,
            min_edges=edges, max_edges=edges, edge_cap=caps.sum(1),
            cap_violations=zeros, illegal_edges=zeros, invalid_weights=zeros,
            invalid_proposals=zeros + (~jnp.isfinite(scale)).sum(),
            initial_max_abs_weight=jnp.abs(W).max((1, 2)), max_abs_weight=jnp.zeros(M)))

    def step(s, _):
        key, ka, kr = jax.random.split(s["key"], 3)
        obs = observe(s["grid"], s["pos"], s["alive"], s["sig"], s["energy"], s["denergy"], c)
        W, x, Wp, surprise = jax.vmap(
            lambda W, x, Wp, o, de, mk: brain_step(
                W, x, Wp, o, de, mk, hebb, eta, plr, g["mod"], pl, c, predictor_plastic=pp)
        )(s["W"], s["x"], s["Wp"], obs, s["denergy"], s['mask'] if structural else mask)
        if structural:
            # The current neural tick/action is already computed; rewiring affects the next tick.
            def event(_):
                draws = jax.vmap(lambda k: jax.random.normal(jax.random.fold_in(k, s['t']), (n,)))(structural_keys)
                return jax.vmap(lambda w, mk, x0, cap, draw, alive: rewire(
                    w, mk, x0, legal, cap, scale, draw, wmax=c.wmax, alive=alive > 0, plastic=plastic)
                )(W, s['mask'], s['x'], caps, draws, s['alive'])

            new_mask = s['mask']
            delta = dict(added=jnp.zeros(M, jnp.int32), removed=jnp.zeros(M, jnp.int32),
                         invalid_proposals=jnp.zeros(M, jnp.int32))
            if plastic:
                W, new_mask, delta = jax.lax.cond((s['t'] > 0) & (s['t'] % INTERVAL == 0), event,
                                                 lambda _: (W, new_mask, delta), operand=None)
            edges = (new_mask > 0).sum((1, 2))
            stats = dict(s['structure'])
            for k in delta:
                stats[k] = stats[k] + delta[k]
            stats.update(final_edges=edges, min_edges=jnp.minimum(stats['min_edges'], edges),
                         max_edges=jnp.maximum(stats['max_edges'], edges),
                         max_abs_weight=jnp.maximum(stats['max_abs_weight'], jnp.abs(W).max((1, 2))))
            stats['cap_violations'] += ((new_mask > 0).sum(2) > caps).sum(1)
            stats['illegal_edges'] += ((new_mask > 0) & ~legal).sum((1, 2))
            stats['invalid_weights'] += ((~jnp.isfinite(W)) | (jnp.abs(W) > c.wmax) |
                                         ((new_mask == 0) & (W != 0))).sum((1, 2))
        logits = 3.0 * x[:, n - O:n - 1]
        act = jax.random.categorical(ka, logits)
        # efference copy: motor neurons "know" which action was actually taken, so the
        # next Hebbian update links what was seen to what was done
        x = x.at[:, n - O:n - 1].set(jax.nn.one_hot(act, O - 1))
        sig = x[:, n - 1] * s["alive"]
        alive = s["alive"]
        pos = (s["pos"] + MOVES[act] * alive[:, None].astype(jnp.int32)) % c.G
        item = s["grid"][pos[:, 0], pos[:, 1]]
        eat = (alive > 0) & (item > 0)
        is_food = eat & (item == food_t)
        is_pois = eat & (item == poison_t)
        e_new = s["energy"] - c.decay * alive + c.gain * is_food - c.gain * is_pois
        e_new = jnp.clip(e_new, 0.0, c.e_max)
        denergy = (e_new - s["energy"]) * alive
        alive_new = alive * (e_new > 0)
        grid = s["grid"].at[pos[:, 0], pos[:, 1]].set(jnp.where(eat, 0, item))
        newc = jax.random.randint(kr, (M, 2), 0, c.G)       # eaten blobs regrow elsewhere
        grid = grid.at[newc[:, 0], newc[:, 1]].set(jnp.where(eat, item, grid[newc[:, 0], newc[:, 1]]))
        half = (s["t"] >= c.T // 2).astype(jnp.int32)
        counts = s["counts"].at[:, half, 0].add(is_food).at[:, half, 1].add(is_pois)
        ns = dict(grid=grid, pos=pos, W=W, x=x, Wp=Wp, energy=e_new, alive=alive_new, sig=sig,
                  denergy=denergy, counts=counts, key=key, t=s["t"] + 1)
        out = dict(surprise=jnp.mean(surprise))
        if structural:
            ns.update(mask=new_mask, structure=stats)
            if record:
                out.update(structural_added=delta['added'].sum(), structural_removed=delta['removed'].sum())
        if record:
            out.update(grid=grid, pos=pos, alive=alive_new, sig=sig, energy=e_new)
        return ns, out

    s, traj = jax.lax.scan(step, carry, None, length=c.T)
    counts = s["counts"]                                     # (M, half, food/poison)
    fitness = jnp.mean(counts[:, :, 0].sum(1) - counts[:, :, 1].sum(1))
    res = dict(fitness=fitness, counts=counts.sum(0), survived=s["alive"].mean(),
               surprise_first=traj["surprise"][: c.T // 2].mean(),
               surprise_last=traj["surprise"][c.T // 2:].mean())
    if structural:
        res['structure'] = jax.tree.map(jnp.mean, s['structure'])
        if record:
            res['structural_state'] = dict(mask=s['mask'], W=s['W'], x=s['x'], row_caps=caps)
    if record:
        res["traj"] = traj
    return res


# ----------------------------------------------------------------------------- evolution
def make_evaluator(c, E, plastic=True, *, predictor_plastic=None, structural=False):
    """Evaluate a population of flat genomes in E worlds each (half with each rule)."""
    _, unravel = ravel_pytree(init_genome(c, jax.random.PRNGKey(0)))
    rules = jnp.arange(E) % 2

    def one(flat, wkeys, dkeys):
        g = unravel(flat)
        r = jax.vmap(lambda wk, dk, ru: rollout(
            g, wk, dk, ru, c, plastic, predictor_plastic=predictor_plastic,
            structural=structural))(wkeys, dkeys, rules)
        return r

    @jax.jit
    def evaluate(pop, wkeys, dkeys):
        return jax.vmap(one, in_axes=(0, None, 0))(pop, wkeys, dkeys)

    return evaluate, unravel


def centered_ranks(f):
    r = jnp.argsort(jnp.argsort(f))
    return r / (f.size - 1) - 0.5


def es_init(c, mean=None, seed=0):
    """Evolution state: the mean genome, Adam moments, step count and RNG key.
    Kept as plain arrays so it can be checkpointed and resumed exactly."""
    key = jax.random.PRNGKey(seed)
    key, k0 = jax.random.split(key)
    flat0, _ = ravel_pytree(init_genome(c, k0))
    mu = flat0 if mean is None else jnp.asarray(mean)
    return dict(mu=mu, m=jnp.zeros_like(mu), v=jnp.zeros_like(mu), t=0, key=key)


def es_step(state, evaluate, pop, E, sigma, lr, l2=0.005):
    """One generation of OpenAI-style ES (antithetic sampling, rank shaping, Adam)."""
    assert pop % 2 == 0
    mu, m_, v_, key = state["mu"], state["m"], state["v"], state["key"]
    key, ke, kw, kd = jax.random.split(key, 4)
    eps = jax.random.normal(ke, (pop // 2, mu.size))
    eps = jnp.concatenate([eps, -eps])
    popu = mu + sigma * eps
    wkeys = jax.random.split(kw, E)                           # same worlds for everyone (fair test)
    dk = jax.random.split(kd, pop // 2 * E).reshape(pop // 2, E, 2)
    dkeys = jnp.concatenate([dk, dk])                         # antithetic twins share development noise
    res = evaluate(popu, wkeys, dkeys)
    if 'structure' in res:
        validate_structure(res['structure'])
    fit = res["fitness"].mean(1)
    w = centered_ranks(fit)
    grad = (w[:, None] * eps).sum(0) / (pop * sigma) - l2 * mu
    b1, b2 = 0.9, 0.999
    t = state["t"] + 1
    m_ = b1 * m_ + (1 - b1) * grad
    v_ = b2 * v_ + (1 - b2) * grad ** 2
    mu = mu + lr * (m_ / (1 - b1 ** t)) / (jnp.sqrt(v_ / (1 - b2 ** t)) + 1e-8)
    cnt = res["counts"].sum((0, 1))                           # (half, food/poison)
    acc = cnt[:, 0] / jnp.maximum(cnt.sum(1), 1)
    row = dict(gen=t - 1, fit_mean=float(fit.mean()), fit_best=float(fit.max()),
               fit_std=float(fit.std()), acc_first=float(acc[0]), acc_second=float(acc[1]),
               survived=float(res["survived"].mean()))
    if 'structure' in res:
        row['structure'] = {k: float(v.mean()) for k, v in res['structure'].items()}
    return dict(mu=mu, m=m_, v=v_, t=t, key=key), row


def format_row(row):
    return (f"gen {row['gen']:4d} | fitness mean {row['fit_mean']:6.2f} best {row['fit_best']:6.2f} | "
            f"food-choice accuracy 1st half {row['acc_first']:.2f} -> 2nd half {row['acc_second']:.2f} | "
            f"survived {row['survived']:.2f}")


def evolve(c, generations=200, pop=128, E=4, sigma=0.1, lr=0.1, seed=0, plastic=True,
           log_every=10, mean=None, *, structural=False):
    """Run ES for a number of generations. Returns (mean genome, unravel, history)."""
    _, unravel = ravel_pytree(init_genome(c, jax.random.PRNGKey(0)))
    evaluate, _ = make_evaluator(c, E, plastic, structural=structural)
    state = es_init(c, mean, seed)
    hist = []
    for gen in range(generations):
        state, row = es_step(state, evaluate, pop, E, sigma, lr)
        hist.append(row)
        if log_every and (gen % log_every == 0 or gen == generations - 1):
            print(format_row(row))
    return state["mu"], unravel, hist


def test_genome(mu, c, n_worlds=64, plastic=True, seed=123, *, predictor_plastic=None, structural=False):
    """Fresh, unseen worlds. Returns averages over n_worlds (half with each rule)."""
    evaluate, _ = make_evaluator(c, n_worlds, plastic, predictor_plastic=predictor_plastic, structural=structural)
    key = jax.random.PRNGKey(seed)
    kw, kd = jax.random.split(key)
    res = evaluate(mu[None], jax.random.split(kw, n_worlds),
                   jax.random.split(kd, n_worlds)[None])
    cnt = res["counts"].sum((0, 1))
    acc = cnt[:, 0] / jnp.maximum(cnt.sum(1), 1)
    result = dict(fitness=float(res["fitness"].mean()), acc_first=float(acc[0]),
                acc_second=float(acc[1]), survived=float(res["survived"].mean()),
                eaten=int(cnt.sum()))
    if structural:
        validate_structure(res['structure'])
        result['structure'] = {k: float(v.mean()) for k, v in res['structure'].items()}
    return result

## 2. Settings
`POP` is the number of genomes per generation, `WORLDS` the number of worlds each genome is tested in (half with each food rule), and `GENERATIONS` how long evolution runs.

In [ ]:
QUICK = False          # True = 2-minute smoke test

c = Cfg()                # world/brain settings live in Cfg (see cell above)
if QUICK:
    POP, WORLDS, GENERATIONS = 16, 2, 5
else:
    POP, WORLDS, GENERATIONS = 128, 8, 200
print("genome size:", genome_size(c), "numbers | neurons per brain:", dims(c)[3])

## 3. Is learning even possible here? (a 1-minute check)
Before evolving anything, we compare three brains in 32 unseen worlds:
- a **random genome**, which should score near chance;
- a **biology-seeded genome**, using the rule nature already found: dopamine-gated Hebbian learning, where links from senses to movement strengthen after an action that raised energy and weaken after one that lowered it;
- the **same biology-seeded brain, frozen**, with no plasticity.

In our test run, the biology-seeded brain's food-choice accuracy rose from about 0.58 to 0.69 within a single life, and freezing it dropped fitness from about 6.3 to 1.1. So the architecture *can* learn. The open question is whether evolution can *find* this rule, and improve on it.

In [ ]:
flat_random, unravel = ravel_pytree(init_genome(c, jax.random.PRNGKey(0)))
flat_bio, _ = ravel_pytree(biology_prior(c))
def report(name, r):
    print(f"{name:22s} | fitness {r['fitness']:5.2f} | accuracy 1st half {r['acc_first']:.2f} -> 2nd half {r['acc_second']:.2f} | survived {r['survived']:.2f}")
report("random genome", test_genome(flat_random, c, n_worlds=32))
report("biology-seeded", test_genome(flat_bio, c, n_worlds=32))
report("biology-seeded, frozen", test_genome(flat_bio, c, n_worlds=32, plastic=False))

## 4. Experiment A: can evolution discover learning from a blank start?
This is the hard, open question. In our small CPU test (32 genomes, 50 generations) it did **not**: fitness stayed near chance. This run is 4× bigger in both population and worlds, and 6× longer. Watch the accuracy columns. If the 2nd-half number climbs above the 1st-half number, evolution has invented learning.

In [ ]:
import time, numpy as np
t0 = time.time()
mu_A, unravel, hist_A = evolve(c, generations=GENERATIONS, pop=POP, E=WORLDS, log_every=10)
print(f"done in {(time.time()-t0)/60:.1f} min")
np.save("seed_genome_A.npy", np.asarray(mu_A))

## 5. Experiment B: can evolution improve on biology?
This run starts from the biology-seeded genome and lets evolution change any of its 155 numbers. Mutations are kept small here (`sigma=0.05`, `lr=0.02`). With bigger steps, our test showed evolution scrambling the prior faster than it could improve it. With these settings, a small CPU test held steady (fitness 6.3 → 6.3 over 20 generations). The question for your GPU run is whether it climbs.

In [ ]:
t0 = time.time()
mu_B, _, hist_B = evolve(c, generations=GENERATIONS, pop=POP, E=WORLDS, log_every=10, mean=flat_bio, sigma=0.05, lr=0.02)
print(f"done in {(time.time()-t0)/60:.1f} min")
np.save("seed_genome_B.npy", np.asarray(mu_B))

## 6. Results

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, h in [("A: blank start", hist_A), ("B: biology start", hist_B)]:
    g = [r["gen"] for r in h]
    ax[0].plot(g, [r["fit_mean"] for r in h], label=name)
    ax[1].plot(g, [r["acc_second"] - r["acc_first"] for r in h], label=name)
ax[0].set(title="Population fitness (food minus poison per agent)", xlabel="generation"); ax[0].legend()
ax[1].axhline(0, ls="--", c="gray", lw=1)
ax[1].set(title="Within-life improvement in accuracy\n(2nd half minus 1st half; > 0 = learning)", xlabel="generation"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. The lesion test, in unseen worlds
Each evolved genome lives in new worlds twice: once normally, once with every synapse frozen. If freezing makes it much worse, the brain depends on learning during its life.

In [ ]:
N = 16 if QUICK else 128
for name, mu in [("A: blank start", mu_A), ("B: biology start", mu_B), ("biology prior (no evolution)", flat_bio)]:
    print(name)
    report("   learning", test_genome(mu, c, n_worlds=N))
    report("   frozen", test_genome(mu, c, n_worlds=N, plastic=False))

## 8. Watch a family live
Green = food and red = poison **in this world** (the agents only see "A" and "B" and must work out which is which). Circles are agents, coloured by the signal they're emitting (blue = −1, red = +1). An × marks an agent that has died.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

def show_life(mu, rule=0, seed=7, step=2):
    g = unravel(mu)
    r = rollout(g, jax.random.PRNGKey(seed), jax.random.PRNGKey(seed + 1), rule, c, True, True)
    tr = jax.tree_util.tree_map(np.asarray, r["traj"])
    food = 1 + rule
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    def draw(t):
        ax.clear()
        grid = tr["grid"][t]
        img = np.ones((c.G, c.G, 3))
        img[grid == food] = [0.35, 0.75, 0.35]
        img[grid == 3 - food] = [0.85, 0.35, 0.35]
        ax.imshow(img)
        for i in range(c.M):
            y, x = tr["pos"][t, i]
            if tr["alive"][t, i]:
                ax.scatter(x, y, s=140, c=[tr["sig"][t, i]], cmap="coolwarm", vmin=-1, vmax=1, edgecolors="k")
            else:
                ax.scatter(x, y, s=80, marker="x", c="k")
        ax.set_title(f"step {t}   energy: " + " ".join(f"{e:.1f}" for e in tr["energy"][t]), fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    anim = animation.FuncAnimation(fig, draw, frames=range(0, c.T, step), interval=90)
    plt.close(fig)
    print("counts (1st half, 2nd half) x (food, poison):\n", np.asarray(r["counts"]))
    return HTML(anim.to_jshtml())

show_life(mu_B, rule=0)   # try mu_A, flat_bio, or rule=1

## 9. Where to take it next
Each of these is a single, testable change:
- **Seed from the fly.** Set the starting `conn` (wiring between type pairs) from FlyWire or MaleCNS connectome statistics instead of a flat −1, and compare with random starts.
- **Sleep.** Every N steps, stop acting and replay recent observations through the prediction loop, then test whether agents forget less when worlds change mid-life.
- **Eligibility traces.** Add a genome-controlled fading memory of recent activity, so rewards that arrive a few steps late can still change the right synapses.
- **Meaningful signals.** Measure whether an agent's signal predicts food nearby. If it does, communication has emerged.
- **Strangers, not family.** Fill each world with agents from different genomes and see whether cooperation survives.